# open streat map buildings

#### how to download:
downloading format is osm.pbf, from https://download.geofabrik.de/

convert it to geoJSON:
download osmium:

    sudo apt-get install osmium-tool

In [ ]:
# Run the osmium command line tool to convert .osm.pbf to GeoJSON first

    osmium export path/to/file.osm.pbf -f geojson -o output.geojson

now run this cell to filter the buildings:

In [ ]:
import geojson

# Load GeoJSON and initialize lists
buildings = []
nodes = set()  # Use a set to avoid duplicates

with open('output.geojson') as f:
    data = geojson.load(f)
    for feature in data['features']:
        # Check if the feature represents a building
        if 'building' in feature['properties']:
            buildings.append(feature)  # Add the building

            # Collect all nodes related to this feature
            if 'coordinates' in feature['geometry']:
                coordinates = feature['geometry']['coordinates']
                
                # Handle different geometry types
                if feature['geometry']['type'] == 'Polygon':
                    for coord in coordinates:
                        nodes.update(tuple(coord_point) for coord_point in coord)
                elif feature['geometry']['type'] == 'MultiPolygon':
                    for polygon in coordinates:
                        for coord in polygon:
                            nodes.update(tuple(coord_point) for coord_point in coord)
                # Add more geometry types as needed

# Save filtered buildings with their nodes
output_geojson = {
    'type': 'FeatureCollection',
    'features': buildings,
    'nodes': list(nodes)  # Convert set back to list
}

with open('filtered_buildings.geojson', 'w') as f:
    geojson.dump(output_geojson, f)


filtered_buildings.geojson format:

In [ ]:
# Example GeoJSON format:
# {
#   "type": "FeatureCollection",
#   ...
# }

now, add it to the project:

In [ ]:
from hera import Project 
from hera import toolkitHome

from hera import *
import geopandas as gpd


proj = Project()

from hera.utils.logging import initialize_logging,with_logger
initialize_logging(
          with_logger("hera.measurements.GIS.vector.buildings.analysis", handlers=['console'], level='INFO', propagate=False)
    )

tk = toolkitHome.getToolkit(toolkitName=toolkitHome.GIS_BUILDINGS, projectName='DOCUMENTATION')

tk.addDataSource(dataSourceName="OSM",
                 resource="/home/shira/filtered_buildings.geojson",
                 dataFormat=tk.datatypes.JSON_DICT,
                 version=(0,0,1),overwrite=True)

doc = tk.getDataSourceDocument(datasourceName="OSM")
data = doc.getData()

## filter area
The function filter_buildings_in_area takes a GeoJSON dictionary, filters the buildings in the desired area, and converts the result to a GeoPandas DataFrame.

In [ ]:
min_lon, min_lat, max_lon, max_lat =34.75937, 32.07038, 34.77344, 32.09400  # Example coordinates

In [ ]:
filtered_bulidings = tk.filter_buildings_in_area(data,min_lon, min_lat, max_lon, max_lat)
filtered_bulidings

## get heights
The function get_buildings_height takes a GeoPandas DataFrame as input and returns the heights of the buildings, along with their names and coordinates.

In [ ]:
buildings_height = tk.get_buildings_height(filtered_bulidings)
buildings_height.dropna()